# Sensor calibration training

Supports both CSV filename formats:
- `log_<distance>cm.csv` for mixed/up+down data in one file
- `log_<distance>cm_<start>_to_<end>.csv` for legacy directional logs
- `log_d6t_<distance>cm_report.csv` for D6T-only refit reports

Direction is parsed only for reference. It is not used as a model feature.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import re
from pprint import pformat

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS_DIR = ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

ENABLE_CLEANING = True
CLEANING_STRICTNESS = 'medium'
TRAIN_D6T_ONLY = True

MIN_TEMP = 20.0
MAX_TEMP = 250.0
MAX_STEP_REF = 8.0
MAX_STEP_SENSOR = 15.0
ROBUST_Z_LIMIT = 4.0
ROLLING_WINDOW = 15
ROLLING_DEV_REF = 5.0
ROLLING_DEV_SENSOR = 10.0
MAX_REMOVED_FRACTION_WARNING = 0.20

SIMPLE_CSV_RE = re.compile(r'^log_(?P<distance>\d+)cm\.csv$')
RANGED_CSV_RE = re.compile(r'^log_(?P<distance>\d+)cm_(?P<start>\d+(?:\.\d+)?)_to_(?P<end>\d+(?:\.\d+)?)\.csv$')
D6T_REPORT_RE = re.compile(r'^log_d6t_(?P<distance>\d+)cm_report\.csv$', re.IGNORECASE)
SENSORS = {'mlx90640': 'mlx90640_max', 'smh01b01': 'smh01b01_max', 'd6t': 'd6t_raw'}
ACTIVE_SENSORS = {'d6t': 'd6t_raw'} if TRAIN_D6T_ONLY else SENSORS
REQUIRED_COLUMNS = ['timestamp', 'reference_temp', 'mlx90640_max', 'smh01b01_max', 'd6t_raw']


In [ ]:
def parse_file_meta(path: Path) -> dict | None:
    report = D6T_REPORT_RE.match(path.name)
    if report:
        return {'source_file': path.name, 'source_path': str(path.relative_to(ROOT)), 'distance_cm': int(report.group('distance')), 'range_start_c': np.nan, 'range_end_c': np.nan, 'direction': 'mixed', 'filename_format': 'd6t_report', 'csv_format': 'd6t_report'}
    simple = SIMPLE_CSV_RE.match(path.name)
    if simple:
        return {'source_file': path.name, 'source_path': str(path.relative_to(ROOT)), 'distance_cm': int(simple.group('distance')), 'range_start_c': np.nan, 'range_end_c': np.nan, 'direction': 'mixed', 'filename_format': 'simple', 'csv_format': 'training'}
    ranged = RANGED_CSV_RE.match(path.name)
    if ranged:
        start = float(ranged.group('start')); end = float(ranged.group('end'))
        return {'source_file': path.name, 'source_path': str(path.relative_to(ROOT)), 'distance_cm': int(ranged.group('distance')), 'range_start_c': start, 'range_end_c': end, 'direction': 'up' if start < end else 'down', 'filename_format': 'ranged', 'csv_format': 'training'}
    return None

def normalize_loaded_csv(data: pd.DataFrame, meta: dict) -> pd.DataFrame:
    if meta['csv_format'] == 'd6t_report':
        missing = {'Time', 'GDM', 'D6T_raw'} - set(data.columns)
        if missing: raise ValueError(f'Missing D6T report columns: {sorted(missing)}')
        out = pd.DataFrame({'timestamp': data['Time'], 'reference_temp': data['GDM'], 'mlx90640_max': np.nan, 'smh01b01_max': np.nan, 'd6t_raw': data['D6T_raw'], 'd6t_calib_old': data['D6T_calib'] if 'D6T_calib' in data.columns else np.nan, 'diff_d6t_report_raw': data['Dif_raw'] if 'Dif_raw' in data.columns else np.nan, 'diff_d6t_report_calib': data['Dif_calib'] if 'Dif_calib' in data.columns else np.nan, 'report_result': data['Result'] if 'Result' in data.columns else np.nan})
    else:
        missing = set(REQUIRED_COLUMNS) - set(data.columns)
        if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')
        out = data[REQUIRED_COLUMNS].copy()
        out['d6t_calib_old'] = np.nan; out['diff_d6t_report_raw'] = np.nan; out['diff_d6t_report_calib'] = np.nan; out['report_result'] = np.nan
    for key, value in meta.items(): out[key] = value
    return out

def discover_csv_files(root: Path) -> list[Path]:
    found=[]; seen=set()
    for path in root.rglob('*.csv'):
        if any(part in {'.git', '.pio', '.venv', '.venv-1', '__pycache__', 'reports'} for part in path.parts): continue
        resolved=path.resolve()
        if resolved in seen: continue
        seen.add(resolved); found.append(path)
    return sorted(found, key=lambda p: str(p.relative_to(root)))

found_files = discover_csv_files(ROOT)
skipped_files=[]; seen_content_hashes={}; frames=[]; file_counts=[]
for path in found_files:
    meta=parse_file_meta(path)
    if meta is None:
        skipped_files.append({'source_path': str(path.relative_to(ROOT)), 'reason': 'Unknown filename format'}); continue
    try:
        content_hash=hashlib.sha256(path.read_bytes()).hexdigest()
        if content_hash in seen_content_hashes:
            skipped_files.append({'source_path': str(path.relative_to(ROOT)), 'reason': f'Duplicate file content: {seen_content_hashes[content_hash]}'}); continue
        seen_content_hashes[content_hash]=str(path.relative_to(ROOT))
        data=normalize_loaded_csv(pd.read_csv(path), meta)
    except pd.errors.EmptyDataError:
        skipped_files.append({'source_path': str(path.relative_to(ROOT)), 'reason': 'Empty file'}); continue
    except Exception as exc:
        skipped_files.append({'source_path': str(path.relative_to(ROOT)), 'reason': f'Parse error: {exc}'}); continue
    if data.empty:
        skipped_files.append({'source_path': str(path.relative_to(ROOT)), 'reason': 'Empty file'}); continue
    frames.append(data); file_counts.append({**meta, 'rows': len(data)})
if not frames: raise RuntimeError('No calibration CSV files found')

df_raw = pd.concat(frames, ignore_index=True)
for col in ['reference_temp', 'mlx90640_max', 'smh01b01_max', 'd6t_raw', 'd6t_calib_old', 'diff_d6t_report_raw', 'diff_d6t_report_calib', 'distance_cm']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
df_raw['timestamp_parsed'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
for sensor, raw_col in SENSORS.items(): df_raw[f'diff_{sensor}_raw'] = df_raw[raw_col] - df_raw['reference_temp']
counts_df = pd.DataFrame(file_counts).sort_values(['distance_cm', 'filename_format', 'source_path'])
skipped_df = pd.DataFrame(skipped_files, columns=['source_path', 'reason'])
report_file_count = int(counts_df['csv_format'].eq('d6t_report').sum()) if len(counts_df) else 0
print(f'Total files found: {len(found_files):,}')
print(f'Total files loaded: {len(counts_df):,}')
print(f'D6T report files loaded: {report_file_count:,}')
print(f'Total rows loaded: {len(df_raw):,}')
print('log_15cm.csv detected:', any(counts_df['source_file'].eq('log_15cm.csv')))
display(counts_df[['source_path', 'csv_format', 'distance_cm', 'direction', 'rows']]); display(skipped_df); display(df_raw.head())


In [ ]:
# Debug: print every CSV file discovered by the recursive scan.
for p in sorted(found_files): print(p)


In [ ]:
# Data checks on raw loaded data.
check_columns = ['timestamp', 'reference_temp', 'mlx90640_max', 'smh01b01_max', 'd6t_raw', 'distance_cm', 'direction', 'csv_format']
display(df_raw[check_columns].isna().sum().rename('nan_count').to_frame())
display(df_raw[['reference_temp', 'mlx90640_max', 'smh01b01_max', 'd6t_raw', 'distance_cm']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)
active_raw_cols = ['reference_temp', 'distance_cm'] + list(ACTIVE_SENSORS.values())
bad = df_raw[(~np.isfinite(df_raw[active_raw_cols]).all(axis=1)) | df_raw['reference_temp'].lt(-40) | df_raw['reference_temp'].gt(350) | df_raw[list(ACTIVE_SENSORS.values())].lt(-40).any(axis=1) | df_raw[list(ACTIVE_SENSORS.values())].gt(400).any(axis=1)]
print(f'Suspicious rows for active sensors: {len(bad):,}'); display(bad.head(20))


## Data Cleaning / Normalization


In [ ]:
def append_reason(reason_series: pd.Series, mask: pd.Series, reason: str) -> pd.Series:
    mask = mask.fillna(False)
    reason_series.loc[mask] = np.where(reason_series.loc[mask].astype(str).str.len().gt(0), reason_series.loc[mask].astype(str) + '|' + reason, reason)
    return reason_series

def finite_mask(data: pd.DataFrame, columns: list[str]) -> pd.Series:
    return pd.Series(np.isfinite(data[columns].to_numpy(dtype=float)).all(axis=1), index=data.index)

def mark_iqr_outliers(values: pd.Series, limit: float = 1.5) -> pd.Series:
    q1 = values.quantile(0.25); q3 = values.quantile(0.75); iqr = q3 - q1
    if not np.isfinite(iqr) or iqr == 0: return pd.Series(False, index=values.index)
    return values.lt(q1 - limit * iqr) | values.gt(q3 + limit * iqr)

def mark_robust_outliers(values: pd.Series, z_limit: float = ROBUST_Z_LIMIT) -> pd.Series:
    values = pd.to_numeric(values, errors='coerce'); valid = values.dropna(); out = pd.Series(False, index=values.index)
    if len(valid) < 8: return out
    median = valid.median(); mad = (valid - median).abs().median()
    if np.isfinite(mad) and mad > 0: return (0.6745 * (values - median) / mad).abs().gt(z_limit).fillna(False)
    return mark_iqr_outliers(values)

def build_clean_dataframe(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    working = data.copy(); reasons = pd.Series('', index=working.index, dtype='object')
    primary_cols = ['reference_temp', 'distance_cm'] + list(ACTIVE_SENSORS.values())
    temp_cols = ['reference_temp'] + list(ACTIVE_SENSORS.values())
    invalid_mask = working[primary_cols].isna().any(axis=1) | ~finite_mask(working, primary_cols)
    reasons = append_reason(reasons, invalid_mask, 'nan_or_non_finite')
    range_mask = pd.Series(False, index=working.index)
    for col in temp_cols: range_mask |= working[col].lt(MIN_TEMP) | working[col].gt(MAX_TEMP)
    reasons = append_reason(reasons, range_mask, 'physical_range')
    ordered = working.sort_values(['source_path', 'timestamp_parsed', 'timestamp'], kind='mergesort')
    step_mask = pd.Series(False, index=working.index); rolling_mask = pd.Series(False, index=working.index)
    for _, group in ordered.groupby('source_path', sort=False):
        for col in temp_cols:
            step_limit = MAX_STEP_REF if col == 'reference_temp' else MAX_STEP_SENSOR
            step_mask.loc[group.index] |= group[col].diff().abs().gt(step_limit).fillna(False)
            rolling_limit = ROLLING_DEV_REF if col == 'reference_temp' else ROLLING_DEV_SENSOR
            rolling_median = group[col].rolling(ROLLING_WINDOW, center=True, min_periods=max(5, ROLLING_WINDOW // 3)).median()
            rolling_mask.loc[group.index] |= (group[col] - rolling_median).abs().gt(rolling_limit).fillna(False)
    reasons = append_reason(reasons, step_mask, 'time_step_spike')
    reasons = append_reason(reasons, rolling_mask, 'rolling_median_outlier')
    residual_mask = pd.Series(False, index=working.index)
    for sensor, raw_col in ACTIVE_SENSORS.items():
        diff_col = f'diff_{sensor}_raw'; working[diff_col] = working[raw_col] - working['reference_temp']
        for _, group in working.groupby('distance_cm', dropna=False): residual_mask.loc[group.index] |= mark_robust_outliers(group[diff_col])
    reasons = append_reason(reasons, residual_mask, 'residual_mad_iqr_outlier')
    working['reject_reason'] = reasons.replace('', np.nan)
    rejected = working[working['reject_reason'].notna()].copy(); clean = working[working['reject_reason'].isna()].copy()
    stats_df = pd.DataFrame([{'rows_raw': int(len(working)), 'rows_clean': int(len(clean)), 'rows_removed': int(len(rejected)), 'removed_pct': float(len(rejected) / len(working) * 100) if len(working) else 0.0}])
    return clean, rejected, stats_df

df_clean, rejected_samples, cleaning_stats = build_clean_dataframe(df_raw)
TRAIN_DF = df_clean if ENABLE_CLEANING else df_raw.copy(); TRAIN_DF_NAME = 'clean' if ENABLE_CLEANING else 'raw'
rejected_export_cols = ['source_path', 'timestamp', 'distance_cm', 'reference_temp', 'mlx90640_max', 'smh01b01_max', 'd6t_raw', 'reject_reason']
rejected_samples[rejected_export_cols].to_csv(REPORTS_DIR / 'rejected_samples.csv', index=False)
df_clean[[c for c in df_clean.columns if c != 'reject_reason']].to_csv(REPORTS_DIR / 'clean_training_data.csv', index=False)
rows_raw = int(cleaning_stats.loc[0, 'rows_raw']); rows_clean = int(cleaning_stats.loc[0, 'rows_clean']); rows_removed = int(cleaning_stats.loc[0, 'rows_removed']); removed_pct = float(cleaning_stats.loc[0, 'removed_pct'])
print('[CLEAN] enabled:', ENABLE_CLEANING); print('[CLEAN] strictness:', CLEANING_STRICTNESS); print('[CLEAN] rows raw:', f'{rows_raw:,}'); print('[CLEAN] rows clean:', f'{rows_clean:,}'); print('[CLEAN] rows removed:', f'{rows_removed:,}'); print('[CLEAN] removed %:', f'{removed_pct:.2f}%')
if rows_raw and rows_removed / rows_raw > MAX_REMOVED_FRACTION_WARNING: print('[CLEAN] Too many rows removed, check thresholds')
removed_by_source = rejected_samples.groupby('source_path').size().rename('rows_removed').reset_index().sort_values('rows_removed', ascending=False)
removed_by_distance = rejected_samples.groupby('distance_cm', dropna=False).size().rename('rows_removed').reset_index().sort_values('distance_cm')
print('[CLEAN] rejected_samples.csv:', REPORTS_DIR / 'rejected_samples.csv'); print('[CLEAN] clean_training_data.csv:', REPORTS_DIR / 'clean_training_data.csv')
display(cleaning_stats); display(removed_by_source); display(removed_by_distance); display(rejected_samples[rejected_export_cols].head(30))


In [ ]:
# Quick plots: sensor raw vs reference, after optional cleaning.
plot_df = TRAIN_DF.sample(min(len(TRAIN_DF), 12000), random_state=7) if len(TRAIN_DF) > 12000 else TRAIN_DF
fig, axes = plt.subplots(1, len(ACTIVE_SENSORS), figsize=(6 * len(ACTIVE_SENSORS), 4), sharey=True)
if len(ACTIVE_SENSORS) == 1: axes = [axes]
for ax, (sensor, col) in zip(axes, ACTIVE_SENSORS.items()):
    sc = ax.scatter(plot_df[col], plot_df['reference_temp'], c=plot_df['distance_cm'], s=4, alpha=0.35, cmap='viridis')
    lo = np.nanmin([plot_df[col].min(), plot_df['reference_temp'].min()]); hi = np.nanmax([plot_df[col].max(), plot_df['reference_temp'].max()])
    ax.plot([lo, hi], [lo, hi], color='black', lw=1, alpha=0.4); ax.set_title(f'{sensor} ({TRAIN_DF_NAME})'); ax.set_xlabel('raw sensor temp')
axes[0].set_ylabel('reference_temp'); fig.colorbar(sc, ax=axes, label='distance_cm'); plt.tight_layout()


In [ ]:
# Raw differences against reference.
for frame in [df_raw, df_clean, TRAIN_DF]:
    for sensor, raw_col in SENSORS.items(): frame[f'diff_{sensor}_raw'] = frame[raw_col] - frame['reference_temp']
diff_cols = [f'diff_{sensor}_raw' for sensor in ACTIVE_SENSORS]
display(TRAIN_DF.groupby(['distance_cm', 'direction'])[diff_cols].agg(['mean', 'std', 'min', 'max']))


## Before / After Cleaning Plots


In [ ]:
plot_source = df_raw['source_path'].value_counts().idxmax()
before = df_raw[df_raw['source_path'].eq(plot_source)].sort_values(['timestamp_parsed', 'timestamp'])
after = df_clean[df_clean['source_path'].eq(plot_source)].sort_values(['timestamp_parsed', 'timestamp'])
fig, axes = plt.subplots(2, 2, figsize=(15, 8)); axes = axes.ravel()
axes[0].plot(before['timestamp_parsed'], before['reference_temp'], '.', ms=2, alpha=0.35, label='raw'); axes[0].plot(after['timestamp_parsed'], after['reference_temp'], '.', ms=2, alpha=0.55, label='clean'); axes[0].set_title(f'reference_temp over time: {plot_source}'); axes[0].legend()
axes[1].plot(before['timestamp_parsed'], before['d6t_raw'], '.', ms=2, alpha=0.35, label='raw'); axes[1].plot(after['timestamp_parsed'], after['d6t_raw'], '.', ms=2, alpha=0.55, label='clean'); axes[1].set_title('d6t_raw over time'); axes[1].legend()
axes[2].plot(before['timestamp_parsed'], before['diff_d6t_raw'], '.', ms=2, alpha=0.35, label='raw'); axes[2].plot(after['timestamp_parsed'], after['diff_d6t_raw'], '.', ms=2, alpha=0.55, label='clean'); axes[2].set_title('diff_d6t_raw over time'); axes[2].legend()
axes[3].hist(df_raw['diff_d6t_raw'].dropna(), bins=80, alpha=0.45, label='raw'); axes[3].hist(df_clean['diff_d6t_raw'].dropna(), bins=80, alpha=0.55, label='clean'); axes[3].set_title('diff_d6t_raw histogram'); axes[3].legend(); plt.tight_layout()


In [ ]:
MODEL_SPECS = [('linear_raw', 1, ['raw']), ('poly2_raw', 2, ['raw']), ('poly3_raw', 3, ['raw']), ('poly2_raw_distance', 2, ['raw', 'distance_cm']), ('poly3_raw_distance', 3, ['raw', 'distance_cm'])]

def feature_frame(data: pd.DataFrame, raw_col: str, features: list[str]) -> pd.DataFrame:
    out = pd.DataFrame(index=data.index)
    for feature in features: out[feature] = data[raw_col] if feature == 'raw' else data[feature]
    return out

def deterministic_holdout(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    order = data.groupby('source_path').cumcount(); test_mask = order.mod(5).eq(0)
    return data.loc[~test_mask].copy(), data.loc[test_mask].copy()

def model_input_data(data: pd.DataFrame, raw_col: str, features: list[str]) -> pd.DataFrame:
    needed = ['reference_temp', raw_col, 'source_path'] + [f for f in features if f != 'raw']
    out = data.dropna(subset=list(dict.fromkeys(needed))).copy(); numeric_needed = ['reference_temp', raw_col] + [f for f in features if f != 'raw']
    return out[np.isfinite(out[numeric_needed]).all(axis=1)]

def fit_pipeline(train: pd.DataFrame, raw_col: str, degree: int, features: list[str]) -> Pipeline:
    model = Pipeline([('poly', PolynomialFeatures(degree=degree, include_bias=True)), ('linear', LinearRegression(fit_intercept=False))])
    model.fit(feature_frame(train, raw_col, features), train['reference_temp']); return model

def evaluate_model_on(data: pd.DataFrame, sensor: str, raw_col: str, name: str, degree: int, features: list[str], label: str) -> tuple[dict, Pipeline]:
    usable = model_input_data(data, raw_col, features); train, test = deterministic_holdout(usable)
    if len(train) == 0 or len(test) == 0: raise RuntimeError(f'Not enough rows to evaluate {sensor}/{name} on {label}')
    model = fit_pipeline(train, raw_col, degree, features); pred = model.predict(feature_frame(test, raw_col, features))
    eval_df = test[['reference_temp', 'distance_cm', 'direction']].copy(); eval_df['pred'] = pred
    by_distance = {int(distance): float(mean_absolute_error(group['reference_temp'], group['pred'])) for distance, group in eval_df.groupby('distance_cm')}
    row = {'dataset': label, 'sensor': sensor, 'model': name, 'degree': degree, 'features': features, 'MAE': float(mean_absolute_error(test['reference_temp'], pred)), 'RMSE': float(np.sqrt(mean_squared_error(test['reference_temp'], pred))), 'R2': float(r2_score(test['reference_temp'], pred)), 'MAE_by_distance': by_distance, 'distance_mae_span': float(max(by_distance.values()) - min(by_distance.values())) if by_distance else np.nan, 'n_coefficients': int(len(model.named_steps['linear'].coef_)), 'rows': int(len(usable))}
    for direction in ['up', 'down', 'mixed']:
        subset = eval_df[eval_df['direction'].eq(direction)]; row[f'MAE_{direction}'] = float(mean_absolute_error(subset['reference_temp'], subset['pred'])) if len(subset) else np.nan
    row['selection_score'] = row['RMSE'] + 0.10 * row['distance_mae_span'] + 0.002 * row['n_coefficients']; return row, model

rows=[]
for dataset_label, dataset in [('raw', df_raw), ('clean', df_clean)]:
    for sensor, raw_col in ACTIVE_SENSORS.items():
        for name, degree, features in MODEL_SPECS: rows.append(evaluate_model_on(dataset, sensor, raw_col, name, degree, features, dataset_label)[0])
all_results = pd.DataFrame(rows).sort_values(['dataset', 'sensor', 'selection_score']); results = all_results[all_results['dataset'].eq(TRAIN_DF_NAME)].copy()
metric_compare = all_results.pivot_table(index=['sensor', 'model'], columns='dataset', values=['MAE', 'RMSE', 'rows'], aggfunc='first').reset_index()
metric_compare.columns = ['_'.join([str(x) for x in col if x]).rstrip('_') for col in metric_compare.columns.to_flat_index()]
metric_compare = metric_compare.rename(columns={'MAE_raw': 'MAE_raw_train', 'RMSE_raw': 'RMSE_raw_train', 'MAE_clean': 'MAE_clean_train', 'RMSE_clean': 'RMSE_clean_train'}); metric_compare['rows_removed'] = rows_removed
metric_compare = metric_compare.sort_values(['sensor', 'MAE_clean_train', 'RMSE_clean_train'])
display(all_results[['dataset', 'sensor', 'model', 'MAE', 'RMSE', 'R2', 'MAE_up', 'MAE_down', 'MAE_mixed', 'MAE_by_distance', 'distance_mae_span', 'n_coefficients', 'rows', 'selection_score']])
display(metric_compare[['sensor', 'model', 'MAE_raw_train', 'RMSE_raw_train', 'MAE_clean_train', 'RMSE_clean_train', 'rows_raw', 'rows_clean', 'rows_removed']])


In [ ]:
def choose_best(group: pd.DataFrame) -> pd.Series:
    best_rmse = group['RMSE'].min(); close = group[group['RMSE'].le(best_rmse * 1.02 + 0.05)].copy()
    return close.sort_values(['n_coefficients', 'selection_score']).iloc[0]

selected_rows=[]
for sensor_name, group in results.groupby('sensor'):
    row = choose_best(group).copy(); row['sensor'] = sensor_name; selected_rows.append(row)
selected = pd.DataFrame(selected_rows).reset_index(drop=True)
display(selected[['sensor', 'model', 'features', 'MAE', 'RMSE', 'R2', 'MAE_by_distance', 'rows']])

if TRAIN_D6T_ONLY and 'd6t_calib_old' in df_raw.columns:
    d6t_compare_base = TRAIN_DF.dropna(subset=['reference_temp', 'd6t_raw', 'distance_cm']).copy()
    d6t_compare_base = d6t_compare_base[np.isfinite(d6t_compare_base[['reference_temp', 'd6t_raw', 'distance_cm']]).all(axis=1)]
    old_rows = d6t_compare_base.dropna(subset=['d6t_calib_old'])
    selected_d6t = selected[selected['sensor'].eq('d6t')].iloc[0]
    new_model = fit_pipeline(model_input_data(TRAIN_DF, 'd6t_raw', list(selected_d6t['features'])), 'd6t_raw', int(selected_d6t['degree']), list(selected_d6t['features']))
    d6t_compare_base['new_calib'] = new_model.predict(feature_frame(d6t_compare_base, 'd6t_raw', list(selected_d6t['features'])))
    d6t_refit_summary = pd.DataFrame([{'rows_train': int(len(d6t_compare_base)), 'rows_removed': rows_removed, 'MAE_raw': float(mean_absolute_error(d6t_compare_base['reference_temp'], d6t_compare_base['d6t_raw'])), 'MAE_old_calib': float(mean_absolute_error(old_rows['reference_temp'], old_rows['d6t_calib_old'])) if len(old_rows) else np.nan, 'MAE_new_calib': float(mean_absolute_error(d6t_compare_base['reference_temp'], d6t_compare_base['new_calib'])), 'model_d6t_selected': selected_d6t['model'], 'd6t_report_files_loaded': report_file_count}])
    display(d6t_refit_summary)


In [ ]:
def load_existing_profiles() -> dict:
    profile_path = ROOT / 'src' / 'calib_profiles.py'
    if not profile_path.exists(): return {}
    spec = importlib.util.spec_from_file_location('existing_calib_profiles', profile_path); module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return dict(getattr(module, 'CALIB_PROFILES', {}))

def export_profile(sensor: str, raw_col: str, model_name: str, degree: int, features: list[str]) -> dict:
    usable = model_input_data(TRAIN_DF, raw_col, features); model = fit_pipeline(usable, raw_col, degree, features)
    poly = model.named_steps['poly']; linear = model.named_steps['linear']
    return {'model_name': model_name, 'degree': int(degree), 'features': features, 'powers': poly.powers_.astype(int).tolist(), 'coefficients': [float(x) for x in linear.coef_], 'uses_direction': False}

profiles = load_existing_profiles() if TRAIN_D6T_ONLY else {}
for _, row in selected.iterrows(): profiles[row['sensor']] = export_profile(row['sensor'], ACTIVE_SENSORS[row['sensor']], row['model'], int(row['degree']), list(row['features']))
cleaning_rules_used = ['nan_or_non_finite', 'physical_range', 'time_step_spike', 'residual_mad_iqr_outlier', 'rolling_median_outlier']
runtime_template = f"""# Polynomial calibration profiles exported from notebooks/train_calibration.ipynb.
# generated_from_clean_data = {bool(ENABLE_CLEANING)}
# cleaning_rules_used = {cleaning_rules_used!r}
# rows_raw = {rows_raw}
# rows_clean = {rows_clean}
# rows_removed = {rows_removed}
#
# Runtime intentionally depends only on the Python standard library.

from __future__ import annotations

import logging
import math

logger = logging.getLogger("calib")

CALIB_GENERATED_FROM_CLEAN_DATA = {bool(ENABLE_CLEANING)}
CALIB_CLEANING_RULES_USED = {cleaning_rules_used!r}
CALIB_ROWS_RAW = {rows_raw}
CALIB_ROWS_CLEAN = {rows_clean}
CALIB_ROWS_REMOVED = {rows_removed}

CALIB_PROFILES = {pformat(profiles, width=100)}

def calibrate(sensor_name: str, raw_value: float, distance_cm: int | float | None, direction=None) -> float:
    profile = CALIB_PROFILES.get(sensor_name)
    try: raw = float(raw_value)
    except (TypeError, ValueError):
        logger.warning("[CALIB] invalid raw value for sensor=%s, using raw value", sensor_name); return raw_value
    if profile is None:
        logger.warning("[CALIB] profile not found, using raw value"); return raw
    if not math.isfinite(raw): return raw
    values = []
    for feature in profile["features"]:
        if feature == "raw": values.append(raw)
        elif feature == "distance_cm":
            if distance_cm is None:
                logger.warning("[CALIB] distance_cm missing for sensor=%s, using raw value", sensor_name); return raw
            values.append(float(distance_cm))
        else:
            logger.warning("[CALIB] unknown feature=%s for sensor=%s, using raw value", feature, sensor_name); return raw
    result = 0.0
    for coefficient, powers in zip(profile["coefficients"], profile["powers"]):
        term = float(coefficient)
        for value, power in zip(values, powers):
            if power: term *= value ** int(power)
        result += term
    return float(result) if math.isfinite(result) else raw
"""
out_path = ROOT / 'src' / 'calib_profiles.py'; out_path.write_text(runtime_template, encoding='utf-8')
print(f'Exported {out_path}'); print('direction_code in exported profiles:', 'direction_code' in runtime_template); print('generated_from_clean_data:', ENABLE_CLEANING); print('cleaning_rules_used:', cleaning_rules_used); print('rows_raw:', rows_raw, 'rows_clean:', rows_clean, 'rows_removed:', rows_removed); print(json.dumps(profiles, indent=2))
